# Laboratorios 1 y 2 fusionados: regresión lineal con PyTorch

**Autor:** Nataniel Mauricio Arapa Estrada  
**Materia:** SIS420 — Inteligencia Artificial I

Este cuadernillo reúne los dos primeros laboratorios porque ambos estudian regresión lineal:

1. **Regresión lineal simple:** años de experiencia → salario.
2. **Regresión lineal multivariable:** características de video → tiempo de transcodificación (`utime`).

La división de datos, normalización, modelos, entrenamiento, solución analítica, inferencia y métricas se realizan con **PyTorch**. Pandas se usa únicamente para leer el archivo TSV y codificar sus columnas categóricas; Matplotlib se usa para visualizar.


## 1. Configuración

El cuadernillo detecta CPU o GPU y fija una semilla para que los resultados sean reproducibles.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32

print(f"PyTorch: {torch.__version__}")
print(f"Dispositivo: {DEVICE}")


In [ ]:
# En Colab se monta Drive porque los cuadernillos originales guardaban allí los datasets.
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def buscar_archivo(nombre, carpeta_lab):
    # Busca un dataset tanto en el repositorio como en las rutas originales de Drive.
    bases = [
        Path.cwd(),
        Path.cwd() / carpeta_lab,
        Path.cwd().parent / carpeta_lab,
        Path("/content/drive/MyDrive/Colab Notebooks/machine learning/datasets"),
        Path("/content/gdrive/MyDrive/Colab Notebooks/machine learning/datasets"),
    ]
    for base in bases:
        candidato = base / nombre
        if candidato.exists():
            return candidato
    rutas = "\n".join(f"  - {base / nombre}" for base in bases)
    raise FileNotFoundError(f"No se encontró {nombre}. Rutas revisadas:\n{rutas}")


def dividir_aleatoriamente(X, y, proporcion_prueba=0.20, semilla=SEED):
    generador = torch.Generator().manual_seed(semilla)
    indices = torch.randperm(len(y), generator=generador)
    corte = int(len(y) * (1 - proporcion_prueba))
    idx_train, idx_test = indices[:corte], indices[corte:]
    return X[idx_train], X[idx_test], y[idx_train], y[idx_test]


def estandarizar_desde_train(X_train, X_test):
    media = X_train.mean(dim=0, keepdim=True)
    desviacion = X_train.std(dim=0, unbiased=False, keepdim=True).clamp_min(1e-8)
    return (X_train - media) / desviacion, (X_test - media) / desviacion, media, desviacion


def metricas_regresion(y_real, y_predicha):
    y_real = y_real.flatten()
    y_predicha = y_predicha.flatten()
    error = y_predicha - y_real
    mse = error.square().mean()
    mae = error.abs().mean()
    rmse = mse.sqrt()
    ss_res = error.square().sum()
    ss_tot = (y_real - y_real.mean()).square().sum().clamp_min(1e-12)
    r2 = 1 - ss_res / ss_tot
    return {
        "MAE": mae.item(),
        "MSE": mse.item(),
        "RMSE": rmse.item(),
        "R2": r2.item(),
    }


def entrenar_regresor(modelo, X_train, y_train, epocas, tasa, lote, semilla=SEED):
    generador = torch.Generator().manual_seed(semilla)
    cargador = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=lote,
        shuffle=True,
        generator=generador,
    )
    criterio = nn.MSELoss()
    optimizador = torch.optim.Adam(modelo.parameters(), lr=tasa)
    historial = []

    for _ in range(epocas):
        modelo.train()
        suma_perdida = 0.0
        for xb, yb in cargador:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizador.zero_grad()
            pred = modelo(xb).squeeze(1)
            perdida = criterio(pred, yb)
            perdida.backward()
            optimizador.step()
            suma_perdida += perdida.item() * len(yb)
        historial.append(suma_perdida / len(X_train))

    return historial


# Parte A — Lab 1: regresión lineal simple

Se conservan los 30 pares `YearsExperience`/`Salary` del dataset usado en el cuadernillo original, pero se expresan directamente como tensores. El modelo es una capa `nn.Linear(1, 1)`, equivalente a:

$$\hat y = wx + b$$

PyTorch calcula los gradientes mediante autograd; por ello ya no hace falta probar miles de combinaciones de $w$ y $b$ con bucles anidados.


In [ ]:
anios = torch.tensor([
    1.1, 1.3, 1.5, 2.0, 2.2, 2.9, 3.0, 3.2, 3.2, 3.7,
    3.9, 4.0, 4.0, 4.1, 4.5, 4.9, 5.1, 5.3, 5.9, 6.0,
    6.8, 7.1, 7.9, 8.2, 8.7, 9.0, 9.5, 9.6, 10.3, 10.5,
], dtype=DTYPE).reshape(-1, 1)

# Se trabaja en miles de dólares para mantener una escala numérica cómoda.
salarios_miles = torch.tensor([
    39.343, 46.205, 37.731, 43.525, 39.891, 56.642, 60.150, 54.445,
    64.445, 57.189, 63.218, 55.794, 56.957, 57.081, 61.111, 67.938,
    66.029, 83.088, 81.363, 93.940, 91.738, 98.273, 101.302, 113.812,
    109.431, 105.582, 116.969, 112.635, 122.391, 121.872,
], dtype=DTYPE)

X_train_a, X_test_a, y_train_a, y_test_a = dividir_aleatoriamente(
    anios, salarios_miles
)

X_train_a_n, X_test_a_n, x_media_a, x_std_a = estandarizar_desde_train(
    X_train_a, X_test_a
)
y_media_a = y_train_a.mean()
y_std_a = y_train_a.std(unbiased=False).clamp_min(1e-8)
y_train_a_n = (y_train_a - y_media_a) / y_std_a

print(f"Ejemplos totales: {len(anios)}")
print(f"Entrenamiento: {len(X_train_a)} | Prueba: {len(X_test_a)}")


In [ ]:
torch.manual_seed(SEED)
modelo_a = nn.Linear(1, 1).to(DEVICE)
historial_a = entrenar_regresor(
    modelo_a,
    X_train_a_n,
    y_train_a_n,
    epocas=800,
    tasa=0.05,
    lote=len(X_train_a_n),
)

modelo_a.eval()
with torch.inference_mode():
    pred_test_a_n = modelo_a(X_test_a_n.to(DEVICE)).cpu().squeeze(1)
    pred_test_a = pred_test_a_n * y_std_a + y_media_a

resultados_a = metricas_regresion(y_test_a, pred_test_a)
print(f"Pérdida normalizada final: {historial_a[-1]:.6f}")
print(f"MAE de prueba:  ${resultados_a['MAE'] * 1000:,.2f}")
print(f"RMSE de prueba: ${resultados_a['RMSE'] * 1000:,.2f}")
print(f"R² de prueba:   {resultados_a['R2']:.4f}")


In [ ]:
# Convertir los parámetros aprendidos a la escala original: salario = w * años + b.
peso_n = modelo_a.weight.detach().cpu().squeeze()
sesgo_n = modelo_a.bias.detach().cpu().squeeze()
w_miles = y_std_a * peso_n / x_std_a.squeeze()
b_miles = y_media_a + y_std_a * sesgo_n - w_miles * x_media_a.squeeze()

print("Parámetros en la escala original")
print(f"w = ${w_miles.item() * 1000:,.2f} por año")
print(f"b = ${b_miles.item() * 1000:,.2f}")

nuevos_anios = torch.arange(6.0, 16.0, dtype=DTYPE).reshape(-1, 1)
nuevos_anios_n = (nuevos_anios - x_media_a) / x_std_a
with torch.inference_mode():
    nuevas_pred_n = modelo_a(nuevos_anios_n.to(DEVICE)).cpu().squeeze(1)
    nuevas_pred = nuevas_pred_n * y_std_a + y_media_a

print("\nPredicciones (los valores mayores a 10.5 años son extrapolaciones):")
for experiencia, salario in zip(nuevos_anios.squeeze(1), nuevas_pred):
    print(f"{experiencia.item():>4.1f} años -> ${salario.item() * 1000:>10,.2f}")


In [ ]:
x_linea = torch.linspace(anios.min().item(), 15.0, 200).reshape(-1, 1)
x_linea_n = (x_linea - x_media_a) / x_std_a
with torch.inference_mode():
    y_linea = (
        modelo_a(x_linea_n.to(DEVICE)).cpu().squeeze(1) * y_std_a + y_media_a
    )

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
ejes[0].plot(historial_a)
ejes[0].set(title="Entrenamiento", xlabel="Época", ylabel="MSE normalizado")
ejes[0].grid(alpha=0.3)

ejes[1].scatter(
    anios.squeeze(1).tolist(), salarios_miles.tolist(), alpha=0.8, label="Datos"
)
ejes[1].plot(x_linea.squeeze(1).tolist(), y_linea.tolist(), color="crimson", label="Modelo")
ejes[1].set(
    title="Salario frente a experiencia",
    xlabel="Años de experiencia",
    ylabel="Salario (miles de $)",
)
ejes[1].legend()
ejes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


# Parte B — Lab 2: regresión lineal multivariable

Se predice `utime` a partir del dataset `transcoding_mesurment.tsv`. Las columnas categóricas se codifican con one-hot; `id` se elimina porque identifica archivos y no representa una característica generalizable.

La separación ocurre **antes** de calcular medias y desviaciones, evitando fuga de información. También se estandariza el objetivo durante el entrenamiento y se devuelve cada predicción a segundos antes de evaluar.


In [ ]:
ruta_transcoding = buscar_archivo("transcoding_mesurment.tsv", "Lab2")
datos_b = pd.read_csv(ruta_transcoding, sep="	").dropna().copy()

y_b = torch.tensor(datos_b.pop("utime").to_numpy(), dtype=DTYPE)
caracteristicas_b = datos_b.drop(columns=["id"], errors="ignore")
caracteristicas_b = pd.get_dummies(caracteristicas_b, dtype="float32")
nombres_b = caracteristicas_b.columns.tolist()
X_b = torch.tensor(caracteristicas_b.to_numpy(dtype="float32"), dtype=DTYPE)

X_train_b, X_test_b, y_train_b, y_test_b = dividir_aleatoriamente(X_b, y_b)
X_train_b_n, X_test_b_n, x_media_b, x_std_b = estandarizar_desde_train(
    X_train_b, X_test_b
)
y_media_b = y_train_b.mean()
y_std_b = y_train_b.std(unbiased=False).clamp_min(1e-8)
y_train_b_n = (y_train_b - y_media_b) / y_std_b

print(f"Dataset: {ruta_transcoding}")
print(f"Filas: {len(X_b):,} | Características tras one-hot: {X_b.shape[1]}")
print(f"Entrenamiento: {len(X_train_b):,} | Prueba: {len(X_test_b):,}")


In [ ]:
torch.manual_seed(SEED)
modelo_b = nn.Linear(X_train_b_n.shape[1], 1).to(DEVICE)
historial_b = entrenar_regresor(
    modelo_b,
    X_train_b_n,
    y_train_b_n,
    epocas=120,
    tasa=0.01,
    lote=1024,
)

modelo_b.eval()
with torch.inference_mode():
    pred_b_n = modelo_b(X_test_b_n.to(DEVICE)).cpu().squeeze(1)
    pred_b = pred_b_n * y_std_b + y_media_b

metricas_b = metricas_regresion(y_test_b, pred_b)
print(f"Pérdida normalizada final: {historial_b[-1]:.6f}")
for nombre, valor in metricas_b.items():
    print(f"{nombre:>4}: {valor:.4f}")


## Solución analítica con `torch.linalg.lstsq`

Para conservar la comparación del Lab 2 con la ecuación normal, se calcula la solución de mínimos cuadrados exclusivamente con PyTorch. `lstsq` es numéricamente más estable que invertir manualmente $X^TX$.


In [ ]:
X_train_b_aug = torch.cat(
    [torch.ones((len(X_train_b_n), 1), dtype=torch.float64), X_train_b_n.double()],
    dim=1,
)
X_test_b_aug = torch.cat(
    [torch.ones((len(X_test_b_n), 1), dtype=torch.float64), X_test_b_n.double()],
    dim=1,
)

theta_b = torch.linalg.lstsq(
    X_train_b_aug,
    y_train_b_n.double().unsqueeze(1),
    driver="gelsd",
).solution
pred_lstsq_b_n = (X_test_b_aug @ theta_b).squeeze(1).float()
pred_lstsq_b = pred_lstsq_b_n * y_std_b + y_media_b
metricas_lstsq_b = metricas_regresion(y_test_b, pred_lstsq_b)

comparacion_b = pd.DataFrame(
    [
        {"Modelo": "nn.Linear + Adam", **metricas_b},
        {"Modelo": "torch.linalg.lstsq", **metricas_lstsq_b},
    ]
)
display(comparacion_b.round(4))


In [ ]:
# Como todas las entradas fueron estandarizadas, |peso| permite comparar su influencia.
pesos_b = modelo_b.weight.detach().cpu().squeeze(0).abs()
k = min(10, len(nombres_b))
indices_top = torch.topk(pesos_b, k=k).indices
importancia_b = pd.DataFrame({
    "característica": [nombres_b[i] for i in indices_top.tolist()],
    "|peso estandarizado|": pesos_b[indices_top].tolist(),
})
display(importancia_b)

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
ejes[0].plot(historial_b)
ejes[0].set_yscale("log")
ejes[0].set(title="Curva de aprendizaje", xlabel="Época", ylabel="MSE normalizado")
ejes[0].grid(alpha=0.3)

ejes[1].scatter(y_test_b.tolist(), pred_b.tolist(), s=10, alpha=0.25)
minimo = min(y_test_b.min().item(), pred_b.min().item())
maximo = max(y_test_b.max().item(), pred_b.max().item())
ejes[1].plot([minimo, maximo], [minimo, maximo], color="crimson", linestyle="--")
ejes[1].set(
    title="Predicción frente al valor real",
    xlabel="utime real (s)",
    ylabel="utime predicho (s)",
)
ejes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Conclusiones

- `nn.Linear` representa exactamente una regresión lineal y PyTorch aprende sus parámetros mediante autograd.
- La normalización se ajusta solo con entrenamiento; el conjunto de prueba permanece realmente no visto.
- La solución `torch.linalg.lstsq` sirve como referencia del óptimo lineal de mínimos cuadrados.
- En el rango superior de `utime` pueden aparecer errores grandes: un modelo lineal no captura por completo interacciones ni valores atípicos del proceso de transcodificación.
